In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

import joblib

In [ ]:
data = pd.read_csv("creditcard.csv")

data.head()
data.shape
data.info
data.describe()

In [ ]:
sns.countplot(x="Class", data=data)

plt.title("Fraud vs Normal Transactions")

plt.show()

In [ ]:
X = data.drop("Class", axis=1)

y = data["Class"]

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print("Class distribution after SMOTE:")
print(np.bincount(y_resampled))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42
)

In [ ]:
input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))

In [ ]:
encoded = Dense(32, activation="relu")(input_layer)

encoded = Dense(16, activation="relu")(encoded)

latent = Dense(8, activation="relu")(encoded)

In [ ]:
decoded = Dense(16, activation="relu")(latent)

decoded = Dense(32, activation="relu")(decoded)

output = Dense(input_dim, activation="sigmoid")(decoded)

In [ ]:
autoencoder = Model(input_layer, output)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

In [ ]:
autoencoder.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=256,
    validation_split=0.2
)

In [ ]:
encoder = Model(input_layer, latent)

X_train_encoded = encoder.predict(X_train)

X_test_encoded = encoder.predict(X_test)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train_encoded, y_train)

In [ ]:
pred = rf_model.predict(X_test_encoded)

In [ ]:
confusion_matrix(y_test, pred)

In [ ]:
print(classification_report(y_test, pred))

In [ ]:
prob = rf_model.predict_proba(X_test_encoded)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, prob)

plt.plot(fpr, tpr)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.show()

In [ ]:
roc_auc_score(y_test, prob)

In [ ]:
joblib.dump(rf_model, "fraud_model.pkl")

joblib.dump(scaler, "scaler.pkl")

joblib.dump(encoder, "encoder_model.pkl")

print("Models saved successfully")